In [ ]:
# Figures for "Lecture: Earthquake Nucleation and Dynamic Rupture"
# ----------------------------------------------------------------
# Original schematic/teaching figures for Module 8.
# These are simplified conceptual diagrams — NOT reproductions of
# published figures.
#
# Output: PNG + PDF files written to ./figures/
#
# Usage:
#   Run all cells, then reference in MyST with:
#     ```{figure} scripts/figures/fig_spring_slider_schematic.png
#     ```

from __future__ import annotations

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

# -------------------------
# Global style (minimal)
# -------------------------
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)


def _save(fig, fname):
    """Save figure as PNG and PDF."""
    for ext in ("png", "pdf"):
        path = os.path.join(OUTDIR, f"{fname}.{ext}")
        fig.savefig(path, bbox_inches="tight", format=ext)
        print(f"Saved: {path}")
    plt.close(fig)

In [ ]:
# =======================================================================
# Figure 1: Spring-slider schematic
# =======================================================================
# A rigid plate on the left drives a block through a spring.
# The block sits on a frictional surface.

fig, ax = plt.subplots(figsize=(7, 3))
ax.set_xlim(-0.5, 8)
ax.set_ylim(-1.5, 3)
ax.set_aspect("equal")
ax.axis("off")

# Ground / frictional surface
ax.plot([-0.5, 8], [0, 0], "k-", lw=2)
# Hatching below ground
for x in np.arange(-0.3, 8, 0.4):
    ax.plot([x, x - 0.3], [0, -0.3], "k-", lw=0.5)

# Rigid plate (left wall)
ax.add_patch(mpatches.Rectangle((0, 0), 0.3, 2.5, fc="0.6", ec="k", lw=1.5))
ax.annotate("Plate", xy=(0.15, 2.7), ha="center", fontsize=10, fontstyle="italic")

# Plate velocity arrow
ax.annotate("", xy=(-0.1, 1.8), xytext=(-0.1, 0.5),
            arrowprops=dict(arrowstyle="->", lw=1.5, color="C0"))
ax.text(-0.45, 1.15, r"$v_{pl}$", fontsize=13, color="C0", ha="center")

# Spring (zigzag between plate and block)
spring_x = np.linspace(0.3, 3.5, 40)
spring_y = 1.0 + 0.3 * np.sin(np.linspace(0, 6 * np.pi, 40))
ax.plot(spring_x, spring_y, "k-", lw=1.5)
ax.text(1.9, 1.7, r"$k$", fontsize=13, ha="center")

# Block
block_x, block_y = 3.5, 0
block_w, block_h = 2.0, 1.5
ax.add_patch(mpatches.Rectangle((block_x, block_y), block_w, block_h,
                                 fc="#FFDEAD", ec="k", lw=1.5))
ax.text(block_x + block_w / 2, block_h / 2, "Block", ha="center", va="center",
        fontsize=11, fontweight="bold")

# Slip arrow below block
ax.annotate("", xy=(6.0, -0.7), xytext=(3.5, -0.7),
            arrowprops=dict(arrowstyle="->", lw=1.5, color="C3"))
ax.text(4.75, -1.1, r"slip $\delta$", fontsize=12, color="C3", ha="center")

# Friction notation
ax.text(4.5, -0.15, r"$\mu_f$", fontsize=12, color="0.4", ha="center", va="top")

# Driving stress equation
ax.text(6.5, 2.3, r"$\tau = k\,(v_{pl}\,t - \delta)$", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="0.7"))

_save(fig, "fig_spring_slider_schematic")

In [ ]:
# =======================================================================
# Figure 2: Weakening vs. elastic unloading — stability diagram
# =======================================================================
# Key figure: shows friction decreasing with slip and elastic unloading
# line.  Stable vs unstable regimes.

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)

delta = np.linspace(0, 3, 200)

# Friction curve (slip-weakening) — same in both panels
tau_s, tau_d, Dc = 1.0, 0.4, 1.0
tau_f = np.where(delta < Dc, tau_s - (tau_s - tau_d) * delta / Dc, tau_d)

for ax, k, label, color, title in zip(
    axes,
    [0.3, 1.2],
    ["Unstable\n(runaway)", "Stable\n(creep)"],
    ["C3", "C0"],
    ["(a) Unstable: weak spring", "(b) Stable: stiff spring"],
):
    # Friction curve
    ax.plot(delta, tau_f, "k-", lw=2.5, label=r"Friction $\tau_f(\delta)$")

    # Elastic unloading line from the peak
    tau_elastic = tau_s - k * delta
    ax.plot(delta, tau_elastic, "--", lw=2, color=color,
            label=rf"Elastic unloading ($k={k}$)")

    # Shade unstable / stable region
    mask = delta < 2.5
    if k < (tau_s - tau_d) / Dc:  # unstable
        fill_mask = (tau_elastic > tau_f) & mask
        ax.fill_between(delta[fill_mask], tau_f[fill_mask], tau_elastic[fill_mask],
                        alpha=0.15, color="C3")
    else:
        fill_mask = (tau_f > tau_elastic) & mask & (delta > 0)
        ax.fill_between(delta[fill_mask], tau_elastic[fill_mask], tau_f[fill_mask],
                        alpha=0.15, color="C0")

    ax.set_xlabel(r"Slip $\delta$", fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9, loc="upper right")
    ax.set_ylim(-0.1, 1.3)
    ax.set_xlim(0, 3)

    # Annotate regime
    ax.text(1.5, 0.15, label, fontsize=12, ha="center", fontstyle="italic",
            color=color, fontweight="bold")

axes[0].set_ylabel(r"Shear stress $\tau$", fontsize=12)

# Instability criterion box
fig.text(0.5, -0.02,
         r"Instability criterion:  $\dfrac{d\tau_f}{d\delta} > k$"
         r"  (friction weakens faster than elastic unloading)",
         ha="center", fontsize=11,
         bbox=dict(boxstyle="round,pad=0.4", fc="lightyellow", ec="0.7"))

_save(fig, "fig_weakening_vs_unloading")

In [ ]:
# =======================================================================
# Figure 3: Slip-weakening friction law
# =======================================================================

fig, ax = plt.subplots(figsize=(5.5, 4))

tau_s, tau_d, Dc = 1.0, 0.4, 1.0
delta = np.linspace(0, 3, 300)
tau = np.where(delta < Dc, tau_s - (tau_s - tau_d) * delta / Dc, tau_d)

ax.plot(delta, tau, "k-", lw=2.5)

# Shade fracture energy G
d_fill = np.linspace(0, Dc, 100)
tau_fill = tau_s - (tau_s - tau_d) * d_fill / Dc
ax.fill_between(d_fill, tau_d, tau_fill, alpha=0.25, color="C1", label=r"Fracture energy $G$")

# Labels
ax.axhline(tau_s, ls=":", color="0.5", lw=1)
ax.axhline(tau_d, ls=":", color="0.5", lw=1)
ax.axvline(Dc, ls=":", color="0.5", lw=1)

ax.text(-0.18, tau_s, r"$\tau_s$", fontsize=13, va="center", ha="right", fontweight="bold")
ax.text(-0.18, tau_d, r"$\tau_d$", fontsize=13, va="center", ha="right", fontweight="bold")
ax.text(Dc, -0.08, r"$D_c$", fontsize=13, ha="center", fontweight="bold")

# Fracture energy formula
ax.text(1.8, 0.82, r"$G = \frac{1}{2}(\tau_s - \tau_d)\,D_c$", fontsize=13,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="0.7"))

ax.set_xlabel(r"Slip $\delta$", fontsize=12)
ax.set_ylabel(r"Shear stress $\tau$", fontsize=12)
ax.set_title("Slip-weakening friction law", fontsize=12)
ax.legend(fontsize=10, loc="center right")
ax.set_xlim(-0.3, 3)
ax.set_ylim(0, 1.2)

_save(fig, "fig_slip_weakening_law")

In [ ]:
# =======================================================================
# Figure 4: Nucleation stages
# =======================================================================
# Four-panel cartoon showing progressive nucleation on a fault.

fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), constrained_layout=True)

stage_labels = [
    "(1) Locked fault",
    "(2) Small slip patch",
    "(3) Critical size",
    "(4) Dynamic rupture",
]

for i, (ax, label) in enumerate(zip(axes, stage_labels)):
    ax.set_xlim(0, 10)
    ax.set_ylim(-2, 2)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(label, fontsize=10)

    # Draw fault line
    ax.plot([0.5, 9.5], [0, 0], "k-", lw=2)

    if i == 0:
        # Fully locked — stress arrows pushing inward
        for x in [2, 4, 6, 8]:
            ax.annotate("", xy=(x, 0.6), xytext=(x, 1.4),
                        arrowprops=dict(arrowstyle="->", lw=1, color="0.5"))
            ax.annotate("", xy=(x, -0.6), xytext=(x, -1.4),
                        arrowprops=dict(arrowstyle="->", lw=1, color="0.5"))
        ax.text(5, -1.8, "Locked", ha="center", fontsize=9, color="0.4")

    elif i == 1:
        # Small slip patch in center
        ax.plot([4, 6], [0, 0], color="C3", lw=5, solid_capstyle="round")
        ax.text(5, 0.5, "slip", ha="center", fontsize=9, color="C3")

    elif i == 2:
        # Expanded patch approaching critical size
        ax.plot([2.5, 7.5], [0, 0], color="C1", lw=5, solid_capstyle="round")
        # Critical size bracket
        ax.annotate("", xy=(2.5, -0.8), xytext=(7.5, -0.8),
                    arrowprops=dict(arrowstyle="<->", lw=1.2, color="C1"))
        ax.text(5, -1.3, r"$L_c$", ha="center", fontsize=12, color="C1",
                fontweight="bold")

    elif i == 3:
        # Dynamic rupture propagating both ways
        ax.plot([1, 9], [0, 0], color="C3", lw=5, solid_capstyle="round")
        ax.annotate("", xy=(0.5, 0), xytext=(2, 0),
                    arrowprops=dict(arrowstyle="->", lw=2, color="C3"))
        ax.annotate("", xy=(9.5, 0), xytext=(8, 0),
                    arrowprops=dict(arrowstyle="->", lw=2, color="C3"))
        ax.text(5, 0.7, "dynamic", ha="center", fontsize=9, color="C3",
                fontweight="bold")

fig.suptitle("Stages of earthquake nucleation", fontsize=12, y=1.02)

_save(fig, "fig_nucleation_stages")

In [ ]:
# =======================================================================
# Figure 5: Energy release rate vs. fracture energy
# =======================================================================
# Shows G(a) increasing with crack half-length a, crossing G_c.

fig, ax = plt.subplots(figsize=(6, 4.5))

a = np.linspace(0.01, 5, 300)

# Energy release rate for a mode-II crack: G ∝ Δσ² a / μ
G_release = 0.3 * a  # simplified linear scaling for schematic

# Fracture energy (constant for slip-weakening)
G_c = 0.6

# Critical size
a_c = G_c / 0.3

ax.plot(a, G_release, "C3-", lw=2.5, label=r"Energy release rate $\mathcal{G}(a)$")
ax.axhline(G_c, color="C0", ls="--", lw=2, label=r"Fracture energy $G_c$")

# Mark critical size
ax.plot(a_c, G_c, "ko", ms=8, zorder=5)
ax.axvline(a_c, color="0.7", ls=":", lw=1)
ax.text(a_c, -0.08, r"$a_c$", fontsize=13, ha="center", fontweight="bold")

# Shade arrest region
ax.fill_between(a[a < a_c], 0, G_release[a < a_c], alpha=0.1, color="C0")
ax.text(a_c / 2, 0.15, "Arrest", fontsize=11, ha="center", color="C0",
        fontstyle="italic")

# Shade propagation region
mask_prop = a > a_c
ax.fill_between(a[mask_prop], G_c, G_release[mask_prop], alpha=0.1, color="C3")
ax.text(3.5, 0.85, "Propagation", fontsize=11, ha="center", color="C3",
        fontstyle="italic")

ax.set_xlabel(r"Crack half-length $a$", fontsize=12)
ax.set_ylabel("Energy rate", fontsize=12)
ax.set_title("Propagation vs. arrest", fontsize=12)
ax.legend(fontsize=10, loc="lower right")
ax.set_xlim(0, 5)
ax.set_ylim(0, 1.5)

_save(fig, "fig_energy_release_vs_fracture")

In [ ]:
print("All nucleation/dynamics figures generated successfully.")